# Derivation: The Daily Update of the Single Index Model

In this derivation, we obtain the exponentially weighted least squares (EWLS) recursion that the trading method uses to update each stock's single index estimates $(\alpha_{i}, \beta_{i}, \sigma_{\varepsilon,i})$ as new trading days arrive. We derive three running sums $(\mathbf{A}_{t}, \mathbf{b}_{t}, c_{t})$ from the weighted normal equations, and solve them for the estimates and the residual standard deviation. Then we show how the prior weight $N_{0}$ and the half-life $H$ set how fast the estimates follow new data.

> __Learning Objectives:__
>
> By the end of this notebook, you should be able to:
> * __Weighted normal equations:__ Derive the EWLS first-order conditions from the exponentially weighted squared-residual loss. Explain why three running sums, a $2 \times 2$ matrix, a 2-vector, and a scalar, are all that the update needs.
> * __The recursion:__ Show how each running sum updates in one step when a new day arrives: shrink it by the decay, then add the new day. Update the estimates without storing past data.
> * __Residual scale and the prior:__ Recover the residual standard deviation from the same running sums, and seed the sums from the 2014 to 2024 fit with weight $N_{0}$. Describe how the half-life $H$ and the prior weight $N_{0}$ trade fast tracking against noisy estimates.

Let's get started!
___

## The Weighted Least-Squares Problem
We want to update the single index estimates $(\alpha_{i},\beta_{i})$ of stock $i$ as new trading days arrive, without storing every past day. On each day $s=1,2,\ldots,t$, we observe the daily growth rate $g_{i,s}$ of the stock and the daily growth rate $g_{m,s}$ of `SPY` (1/yr). We write the target as $y_{s}=g_{i,s}$, the regressors as $\mathbf{x}_{s}=[1,\;g_{m,s}]^{\top}$, and the estimates as $\boldsymbol{\theta}_{i}=[\alpha_{i},\;\beta_{i}]^{\top}$.

The single index model of the stock is $y_{s}=\mathbf{x}_{s}^{\top}\boldsymbol{\theta}_{i}+\varepsilon_{i,s}$. On day $t$, we give day $s$ the weight $\phi^{\,t-s}$, where the decay $\phi=2^{-1/H}$ halves the weight of a day every $H$ trading days, the half-life. The EWLS estimate minimizes the weighted sum of squared residuals $L_{t}$, and its gradient is zero at the estimate:

$$
\begin{align*}
L_{t}(\boldsymbol{\theta}) &= \sum_{s=1}^{t}\phi^{\,t-s}\left(y_{s}-\mathbf{x}_{s}^{\top}\boldsymbol{\theta}\right)^{2} && \text{weighted sum of squared residuals}\\
\nabla_{\boldsymbol{\theta}}L_{t} &= -2\sum_{s=1}^{t}\phi^{\,t-s}\,\mathbf{x}_{s}\left(y_{s}-\mathbf{x}_{s}^{\top}\boldsymbol{\theta}\right) = \mathbf{0} && \text{first-order condition}\\
\underbrace{\left(\sum_{s=1}^{t}\phi^{\,t-s}\,\mathbf{x}_{s}\mathbf{x}_{s}^{\top}\right)}_{\mathbf{A}_{t}}\boldsymbol{\theta} &= \underbrace{\sum_{s=1}^{t}\phi^{\,t-s}\,\mathbf{x}_{s}\,y_{s}}_{\mathbf{b}_{t}} && \text{weighted normal equations}
\end{align*}
$$

We also keep a third running sum, $c_{t}=\sum_{s=1}^{t}\phi^{\,t-s}y_{s}^{2}$, for the residual standard deviation. The three running sums $(\mathbf{A}_{t},\mathbf{b}_{t},c_{t})$ hold everything that the update needs from the past days. The normal equations have one solution once $g_{m,s}$ has varied over the weighted days, which makes $\mathbf{A}_{t}$ invertible.

___

## The Daily Update and the Estimates
From day $t-1$ to day $t$, the weight $\phi^{\,(t-1)-s}$ of every past day shrinks by the factor $\phi$ to $\phi^{\,t-s}$, and the new day enters with weight $\phi^{0}=1$. Splitting each running sum into the past days and the new day gives the daily update:

$$
\begin{align*}
\mathbf{A}_{t} &= \sum_{s=1}^{t-1}\phi^{\,t-s}\,\mathbf{x}_{s}\mathbf{x}_{s}^{\top} + \mathbf{x}_{t}\mathbf{x}_{t}^{\top} && \text{past days and the new day}\\
&= \phi\sum_{s=1}^{t-1}\phi^{\,(t-1)-s}\,\mathbf{x}_{s}\mathbf{x}_{s}^{\top} + \mathbf{x}_{t}\mathbf{x}_{t}^{\top} && \text{factor out } \phi\\
&= \boxed{\phi\,\mathbf{A}_{t-1} + \mathbf{x}_{t}\mathbf{x}_{t}^{\top}} && \text{definition of } \mathbf{A}_{t-1}\\
\mathbf{b}_{t} &= \boxed{\phi\,\mathbf{b}_{t-1} + \mathbf{x}_{t}\,y_{t}} && \text{same steps}\\
c_{t} &= \boxed{\phi\,c_{t-1} + y_{t}^{2}} && \text{same steps}
\end{align*}
$$

Each day, the update multiplies the three running sums by $\phi$ and adds the new day. We never store a past day, and each day costs the same small amount of work. Solving the weighted normal equations gives the updated estimates $\hat{\boldsymbol{\theta}}_{i,t}=\mathbf{A}_{t}^{-1}\mathbf{b}_{t}$, which the lecture writes as $\alpha_{i}(t)$ and $\beta_{i}(t)$. Each entry of $\mathbf{A}_{t}$ and $\mathbf{b}_{t}$ is a weighted sum over the days: $S_{w}$ adds the weights, and $S_{wm}$, $S_{wmm}$, $S_{wy}$, and $S_{wmy}$ add $g_{m,s}$, $g_{m,s}^{2}$, $y_{s}$, and $g_{m,s}\,y_{s}$. Because $\mathbf{A}_{t}$ is $2\times 2$, Cramer's rule gives the ordinary least-squares slope and intercept, with weighted sums in place of plain sums:

$$
\begin{align*}
\mathbf{A}_{t} &= \begin{bmatrix} S_{w} & S_{wm} \\ S_{wm} & S_{wmm} \end{bmatrix}, \qquad \mathbf{b}_{t} = \begin{bmatrix} S_{wy} \\ S_{wmy} \end{bmatrix} && \text{entries of the running sums}\\
\hat{\beta}_{i,t} &= \boxed{\frac{S_{w}\,S_{wmy} - S_{wm}\,S_{wy}}{S_{w}\,S_{wmm} - S_{wm}^{2}}} && \text{Cramer's rule}\\
\hat{\alpha}_{i,t} &= \boxed{\frac{S_{wy} - \hat{\beta}_{i,t}\,S_{wm}}{S_{w}}} && \text{first normal equation}
\end{align*}
$$

The residual standard deviation comes from the same running sums. We expand the weighted sum of squared residuals at the estimates, and divide it by the total weight $S_{w}$:

$$
\begin{align*}
\sum_{s=1}^{t}\phi^{\,t-s}\left(y_{s}-\mathbf{x}_{s}^{\top}\hat{\boldsymbol{\theta}}_{i,t}\right)^{2} &= c_{t} - 2\,\hat{\boldsymbol{\theta}}_{i,t}^{\top}\mathbf{b}_{t} + \hat{\boldsymbol{\theta}}_{i,t}^{\top}\mathbf{A}_{t}\,\hat{\boldsymbol{\theta}}_{i,t} && \text{expand the square}\\
&= c_{t} - \hat{\boldsymbol{\theta}}_{i,t}^{\top}\mathbf{b}_{t} && \mathbf{A}_{t}\,\hat{\boldsymbol{\theta}}_{i,t}=\mathbf{b}_{t}\\
\hat{\sigma}_{\varepsilon,i,t} &= \boxed{\sqrt{\frac{c_{t} - \hat{\boldsymbol{\theta}}_{i,t}^{\top}\mathbf{b}_{t}}{S_{w}}}} && \text{divide by } S_{w}\text{, take the square root}
\end{align*}
$$

This is the weighted least-squares variance estimate without the degrees-of-freedom correction. With the trading method's settings, the correction would change the estimate by less than two percent.

___

## Seeding the Prior and the Half-Life
Before the first new day, we seed the running sums as if the 2014 to 2024 fit had been seen on $N_{0}$ days. The fit gives the estimates $\boldsymbol{\theta}_{i,0}$, the residual standard deviation $\sigma_{\varepsilon,i,0}$, and the regressor moments $\mathbf{M}=\mathbb{E}\left[\mathbf{x}\mathbf{x}^{\top}\right]$ of the 2014 to 2024 data. The seed is:

$$
\mathbf{A}_{0} = N_{0}\,\mathbf{M}, \qquad \mathbf{b}_{0} = \mathbf{A}_{0}\,\boldsymbol{\theta}_{i,0}, \qquad c_{0} = \boldsymbol{\theta}_{i,0}^{\top}\mathbf{b}_{0} + N_{0}\,\sigma_{\varepsilon,i,0}^{2}
$$

Solving against the seed alone returns $\hat{\boldsymbol{\theta}}_{i,0}=\boldsymbol{\theta}_{i,0}$ and $\hat{\sigma}_{\varepsilon,i,0}=\sigma_{\varepsilon,i,0}$. The update starts at the 2014 to 2024 fit and moves only when new data arrives. The weight of the seed decays like the weight of any other day, to $N_{0}\,\phi^{k}$ after $k$ trading days. A large prior weight $N_{0}$ keeps the 2014 to 2024 fit in charge longer. A small $N_{0}$ lets the first new days move the estimates a lot.

A short half-life, such as $H=21$ days (one trading month), gives estimates that follow changes in the stock quickly but are noisy from day to day. A long half-life, such as $H=126$ days (six months), gives smooth estimates that are slow to follow changes. The trading method uses $H=63$ days (one trading quarter), between the two, and $N_{0}=63$.

___

## Summary

The EWLS recursion compresses the whole weighted history of stock $i$'s single index regression into three running sums $(\mathbf{A}_{t}, \mathbf{b}_{t}, c_{t})$. Each trading day, the update shrinks every sum by the decay $\phi$ and adds the new day. The estimates $\hat{\boldsymbol{\theta}}_{i,t}$ come from a $2 \times 2$ solve, and the residual standard deviation $\hat{\sigma}_{\varepsilon,i,t}$ comes from the same sums.

> __Key Takeaways:__
>
> * __Three running sums are enough:__ The weighted normal equations show that $(\mathbf{A}_{t}, \mathbf{b}_{t}, c_{t})$ are the only quantities that the EWLS estimate needs from the weighted history. We never store a past day.
> * __One step per day:__ Moving from day $t-1$ to day $t$ multiplies every running sum by $\phi = 2^{-1/H}$ and adds the new day. The same rule applies to all three sums, and the estimates and the residual standard deviation follow from a $2 \times 2$ solve.
> * __The prior weight and the half-life are the settings:__ The prior weight $N_{0}$ sets how strongly the 2014 to 2024 fit anchors the estimates before new data arrives. The half-life $H$ sets how fast that anchor and every later day fade.

___